

---

###  **General Pipeline Summary**

1. **Preprocessing & Caching**  
   Convert all audio files to **mel spectrograms** and cache them for fast access.

2. **Data Augmentation**  
   Use **SpecAugment-inspired techniques**: time stretching, frequency & time masking, and noise.

3. **Model Training (Main)**  
   - **K-Fold cross-validation** (5 folds)  
   - CNN with **Squeeze-and-Excite blocks**  
   - Hyperparameter tuning with **KerasTuner (Hyperband)**

4. **Binary Classifier** *(for class 5 vs 7)*  
   Separate model trained to fix confusion between class 5 and 7 using a focused binary dataset.

5. **Testing & Ensemble**  
   - All fold models predict on test set  
   - **Majority vote** for ensemble  
   - Patch ensemble results using **binary classifier** to improve class 5 predictions

6. **Evaluation**  
   Accuracy and classification reports are printed for all components.

---

# TODO :

## Areas for Improvement

1. **Code Organization**:
   - Some naming inconsistencies (e.g., *load*and_process instead of load_and_process)
   - Function definitions could be better grouped by purpose

2. **Binary Model Integration**:
   - While clever, the binary model approach creates two points of potential failure
   - It's unclear how you combine decisions between main and binary models

3. **Model Architecture**:
   - Consider experimenting with different main architectures (EfficientNet, MobileNetV3)
   - The channel dimensions (32→64→128) could be expanded for more expressivity

4. **Error Handling**:
   - Some error handling is present, but could be more systematic
   - Missing handling for potential failures in model ensemble phase

## Recommendations

1. **Ensemble Approach**:
   - Instead of a separate binary model, consider training multiple models with different architectures
   - Use soft voting (average probabilities) instead of hard voting for more nuanced predictions

2. **Feature Engineering**:
   - Try additional acoustic features like MFCCs, spectral contrast, or chroma features
   - Consider multi-input models that use different feature representations

3. **Advanced Techniques**:
   - Implement mixup or cutmix augmentation strategies
   - Add learning rate warmup at the beginning of training
   - Try focal loss instead of standard categorical cross-entropy

4. **Debugging Aid**:
   - Add confusion matrix visualization for each fold and ensemble
   - Implement T-SNE/UMAP visualization of embeddings to see class separation

5. **Binary Model Alternative**:
   - Instead of a separate binary model, consider a hierarchical classification approach
   - Or use a single model with a custom loss that penalizes class 5-7 confusion more heavily

---

### Middle-ground alternatives for ensemble:

1. **Lightweight architectural variations**: Instead of completely different architectures, try variations of your current CNN model with different depths, filter configurations, or attention mechanisms. This gives you architectural diversity without the full cost.

2. **Single architecture with different input features**: Train the same architecture on different input representations (mel spectrograms with different parameters, MFCCs, spectral contrast features, etc.).

3. **Snapshot ensembles**: Train a single model but save its weights at different points during training, especially after learning rate changes. This gives you model diversity at little extra cost.

4. **Two-architecture approach**: If you're concerned about the class 5 vs 7 issue, perhaps use your 5-fold ensemble for the main model and a single specialized model with a different architecture for the problematic classes.

For your specific situation, I'd recommend sticking with your K-fold approach for now since it's already working well, and only exploring multi-architecture ensembles if:

1. You have sufficient computational resources
2. You've exhausted other optimization strategies
3. The performance improvement is critical for your application

The 5-fold ensemble with a specialized binary classifier for classes 5 and 7 seems like a good balance of performance and practicality for your current needs.
---

In [ ]:
!pip install -q numpy
!pip install -q pandas
!pip install -q librosa
!pip install -q tensorflow
!pip install -q scikit-learn
!pip install -q keras-tuner
!pip install -q tensorflow.keras.callbacks
!pip install -q tensorflow.keras


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
import keras_tuner as kt
from sklearn.model_selection import train_test_split


# Data & Audio settings
SR = 22050
N_MELS = 256
FMAX = 8000
FIXED_TIME_STEPS = 150
NUM_CLASSES = 10

# Training parameters
BATCH_SIZE = 32
EPOCHS = 50
N_FOLDS = 5
SEED = 42

# Paths
TRAIN_CSV_PATH = "dataset/Train.csv"
TRAIN_AUDIO_DIR = "dataset/Train/"
TEST_CSV_PATH = "dataset/Test_Public.csv"
TEST_AUDIO_DIR = "dataset/Test_Public/"
CACHE_DIR = "dataset/mel_cache"  # Directory for caching audio features

# Set random seed
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Create cache directory if needed
os.makedirs(CACHE_DIR, exist_ok=True)

# Load and Inspect Metadata
df = pd.read_csv(TRAIN_CSV_PATH)
df["filepath"] = df["file_name"].apply(lambda x: os.path.join(TRAIN_AUDIO_DIR, x))

missing = [f for f in df["filepath"] if not os.path.exists(f)]
print(f"Missing audio files: {len(missing)}")
if missing:
    print(missing[:5])  # Show some samples


file_paths = df["filepath"].values
labels = df["classID"].values
assert len(file_paths) == len(labels), "Mismatch in paths and labels"

assert labels.min() == 0 and labels.max() == NUM_CLASSES - 1, "Label range mismatch!"
assert set(np.unique(labels)) == set(range(NUM_CLASSES)), "Missing some classes!"

# Optional: check distribution
for cls, count in pd.Series(labels).value_counts().sort_index().items():
    print(f"Class {cls}: {count} samples")

Missing audio files: 0
Class 0: 800 samples
Class 1: 368 samples
Class 2: 800 samples
Class 3: 800 samples
Class 4: 800 samples
Class 5: 801 samples
Class 6: 291 samples
Class 7: 828 samples
Class 8: 769 samples
Class 9: 800 samples


In [3]:
def load_and_cache_audio(file_path):
    """Load audio and cache mel spectrogram for faster access"""
    cache_file = os.path.join(
        CACHE_DIR,
        os.path.basename(file_path).replace('.wav', '.npy')
    )

    # If cache exists and loads correctly, use it
    if os.path.exists(cache_file):
        try:
            mel_db = np.load(cache_file)
           # print(f"[SKIP] Using cached: {os.path.basename(cache_file)}")
            return mel_db
        except Exception:
            print(f"[WARN] Failed to load cache: {os.path.basename(cache_file)} — Regenerating.")

    # Generate mel spectrogram
    try:
        y, sr = librosa.load(file_path, sr=SR)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, fmax=FMAX)
        mel_db = librosa.power_to_db(mel, ref=np.max)

        # Normalize
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

        # Add channel dimension
        mel_db = np.expand_dims(mel_db, axis=-1)

        # Cache the result
        np.save(cache_file, mel_db)
        #print(f"[NEW] Cached: {os.path.basename(cache_file)}")

        return mel_db
    except Exception as e:
        print(f"[ERROR] Failed to process {file_path}: {e}")
        return np.zeros((N_MELS, FIXED_TIME_STEPS, 1))


In [4]:
from tqdm import tqdm
import os

def cache_all_audio(file_paths):
    print(" Pre-caching mel spectrograms...\n")
    skipped, created, failed = 0, 0, 0

    for path in tqdm(file_paths):
        cache_file = os.path.join(CACHE_DIR, os.path.basename(path).replace('.wav', '.npy'))

        # Check if already cached before calling the function
        already_cached = os.path.exists(cache_file)

        result = load_and_cache_audio(path)

        if result is None or np.count_nonzero(result) == 0:
            failed += 1
        elif already_cached:
            skipped += 1
        else:
            created += 1

    print(f"\n Done caching.")
    print(f" Created: {created}")
    print(f" Skipped: {skipped}")
    print(f" Failed:  {failed}")
cache_all_audio(file_paths)


 Pre-caching mel spectrograms...



 12%|█▏        | 831/7057 [00:48<03:11, 32.56it/s] /usr/local/lib/python3.10/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1103
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1323
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1523
  warnings.warn(
100%|██████████| 7057/7057 [04:44<00:00, 24.80it/s]


 Done caching.
 Created: 7057
 Skipped: 0
 Failed:  0


### Inspect Mels


In [ ]:
import random

def inspect_random_cached_mels(n=4):
    cached_files = [f for f in os.listdir(CACHE_DIR) if f.endswith('.npy')]
    if len(cached_files) == 0:
        print("No cached mel spectrograms found.")
        return

    print(f"\nInspecting {n} random cached mel spectrograms:\n")

    for fname in random.sample(cached_files, min(n, len(cached_files))):
        path = os.path.join(CACHE_DIR, fname)
        try:
            mel = np.load(path)
            print(f" {fname}")
            print(f"   Shape     : {mel.shape}")
            print(f"   Min/Max   : {mel.min():.4f} / {mel.max():.4f}")
            print(f"   Non-zero% : {np.count_nonzero(mel) / mel.size * 100:.2f}%")
            print("---------------")
        except Exception as e:
            print(f" Failed to load {fname}: {e}")

# Run it
inspect_random_cached_mels()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import random

def visualize_random_cached_mels(n=4):
    cached_files = [f for f in os.listdir(CACHE_DIR) if f.endswith('.npy')]
    if len(cached_files) == 0:
        print(" No cached mel spectrograms found.")
        return

    selected_files = random.sample(cached_files, min(n, len(cached_files)))

    plt.figure(figsize=(15, 4))
    for i, fname in enumerate(selected_files):
        mel = np.load(os.path.join(CACHE_DIR, fname))

        # Squeeze last dimension to get (128, time) for plotting
        mel = np.squeeze(mel, axis=-1)

        plt.subplot(1, n, i + 1)
        plt.imshow(mel, origin='lower', aspect='auto', cmap='magma')
        plt.title(fname)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Run it
visualize_random_cached_mels()


# Binary Classification Model Data Paths

In [5]:
def get_binary_class_dataset(file_paths, labels, class_a=5, class_b=7, shuffle=True):
    binary_paths = []
    binary_labels = []

    for path, label in zip(file_paths, labels):
        if label == class_a:
            binary_paths.append(path)
            binary_labels.append(1)
        elif label == class_b:
            binary_paths.append(path)
            binary_labels.append(0)

    binary_paths = np.array(binary_paths)
    binary_labels = np.array(binary_labels)

    if shuffle:
        idx = np.random.permutation(len(binary_paths))
        binary_paths = binary_paths[idx]
        binary_labels = binary_labels[idx]

    print(f" Binary dataset created: {len(binary_paths)} samples (1s: {np.sum(binary_labels)}, 0s: {np.sum(binary_labels == 0)})")

    return binary_paths, binary_labels


In [6]:
# TensorFlow-optimized augmentation function
@tf.function
def tf_augment_spectrogram(spectrogram):
    orig_shape = tf.shape(spectrogram)
    height = orig_shape[0]
    width = orig_shape[1]

    # === Time Stretch ===
    stretch_factor = tf.random.uniform([], 0.8, 1.2)
    new_width = tf.cast(tf.cast(width, tf.float32) * stretch_factor, tf.int32)
    x = tf.image.resize(spectrogram, [height, new_width])
    x = tf.image.resize(x, [height, width])

    # === Frequency Mask ===
    freq_mask_param = tf.cast(tf.cast(height, tf.float32) * 0.15, tf.int32)
    f0 = tf.random.uniform([], 0, height - freq_mask_param, dtype=tf.int32)
    freq_mask = tf.ones((freq_mask_param, width, 1), dtype=tf.float32)
    freq_pad = [[f0, height - f0 - freq_mask_param], [0, 0], [0, 0]]
    freq_mask = tf.pad(freq_mask, freq_pad, constant_values=0.0)
    x = x * (1.0 - freq_mask)

    # === Time Mask (NEW) ===
    time_mask_param = tf.cast(tf.cast(width, tf.float32) * 0.15, tf.int32)
    t0 = tf.random.uniform([], 0, width - time_mask_param, dtype=tf.int32)
    time_mask = tf.ones((height, time_mask_param, 1), dtype=tf.float32)
    time_pad = [[0, 0], [t0, width - t0 - time_mask_param], [0, 0]]
    time_mask = tf.pad(time_mask, time_pad, constant_values=0.0)
    x = x * (1.0 - time_mask)

    # === Add Gaussian Noise ===
    noise = tf.random.normal(shape=tf.shape(x), mean=0.0, stddev=0.01)
    x = x + noise

    # === Normalize ===
    x_min = tf.reduce_min(x)
    x_max = tf.reduce_max(x)
    x = (x - x_min) / (x_max - x_min + 1e-6)

    return x



# Improved hybrid dataset creation
def create_hybrid_dataset(file_paths, labels, batch_size=32, augment=True):
    """Create dataset with cached features but on-the-fly augmentation"""
    
    def process_path(file_path, label):
        def _load_and_process(path):
            path = path.numpy().decode("utf-8")  
            mel_db = load_and_cache_audio(path)
        
            if mel_db.shape[0] != N_MELS:
                print(f"[WARN] {path}: Expected {N_MELS} mel bins, got {mel_db.shape[0]}. Skipping.")
                mel_db = np.zeros((N_MELS, FIXED_TIME_STEPS), dtype=np.float32)
        
            if mel_db.shape[1] != FIXED_TIME_STEPS:
                mel_db = librosa.util.fix_length(np.squeeze(mel_db, axis=-1), size=FIXED_TIME_STEPS, axis=1)
                mel_db = np.expand_dims(mel_db, axis=-1)
            elif mel_db.ndim == 2:
                mel_db = np.expand_dims(mel_db, axis=-1)
        
            return mel_db.astype(np.float32)

        
        # Pass file_path into the py_function call
        mel_db = tf.py_function(
            func=_load_and_process,
            inp=[file_path],
            Tout=tf.float32
        )
    
        mel_db = tf.ensure_shape(mel_db, [N_MELS, FIXED_TIME_STEPS, 1])
        return mel_db, tf.cast(label, tf.int32)


    # Create base dataset
    dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)

    # Optional: apply augmentation
    if augment:
        dataset = dataset.map(
            lambda x, y: (
                tf.cond(
                    tf.random.uniform([], 0, 1) < 0.7,
                    lambda: tf_augment_spectrogram(x),
                    lambda: x
                ),
                y
            ),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    # Shuffle, batch, prefetch
    dataset = dataset.shuffle(buffer_size=min(1000, len(file_paths)))
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


In [ ]:
# Grab a small sample to test
test_file_paths = file_paths[:16]
test_labels = labels[:16]

# Create the hybrid dataset (augmentation ON for testing)
test_ds = create_hybrid_dataset(test_file_paths, test_labels, batch_size=4, augment=True)

# Take one batch and inspect
for x_batch, y_batch in test_ds.take(1):
    print("Loaded batch successfully!")
    print("Spectrogram batch shape:", x_batch.shape)  # Should be (4, 128, 150, 1)
    print("Labels shape:", y_batch.shape)              # Should be (4,)
    print("Label values:", y_batch.numpy())


# SE Funtion Used In Both Models

In [7]:
def squeeze_excite_block(input_tensor, ratio=16, dropout_rate=0.1):
    """
    Enhanced SE block with dynamic shape support and optional dropout.
    
    Args:
        input_tensor: 4D tensor (batch, height, width, channels)
        ratio: Reduction ratio
        dropout_rate: Dropout between Dense layers (optional)

    Returns:
        Tensor with channel attention applied
    """
    # Dynamic-safe way to get channel dimension
    filters = input_tensor.shape[-1]
    if filters is None:
        filters = tf.shape(input_tensor)[-1]

    # Squeeze: Global spatial information
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Reshape((1, 1, filters))(se)
    
    # Excitation: Bottleneck -> Non-linearity -> Attention
    se = layers.Dense(filters // ratio, activation='relu', kernel_initializer='he_normal')(se)
    
    # Optional: Dropout for regularization
    if dropout_rate > 0:
        se = layers.Dropout(dropout_rate)(se)
    
    se = layers.Dense(filters, activation='sigmoid', kernel_initializer='he_normal')(se)
    
    # Scale the original input
    return layers.multiply([input_tensor, se])

# Binary Model

In [8]:
def build_binary_model(hp):
    lr = hp.Float("learning_rate", 1e-4, 1e-3, sampling="log")
    dropout = hp.Float("dropout", 0.3, 0.6, step=0.1)
    l2_val = hp.Float("l2", 1e-5, 1e-3, sampling="log")
    se_ratio = hp.Choice("se_ratio", [8, 16])
    se_dropout = hp.Float("se_dropout", 0.05, 0.2, step=0.05)

    inputs = tf.keras.Input(shape=(N_MELS, FIXED_TIME_STEPS, 1))

    # Initial block
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 1
    residual = x
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = squeeze_excite_block(x, ratio=se_ratio, dropout_rate=se_dropout)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 2
    residual = layers.Conv2D(64, (1, 1), padding='same')(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = squeeze_excite_block(x, ratio=se_ratio, dropout_rate=se_dropout)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 3
    residual = layers.Conv2D(128, (1, 1), padding='same')(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = squeeze_excite_block(x, ratio=se_ratio, dropout_rate=se_dropout)
    x = layers.MaxPooling2D((2, 2))(x)

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(l2_val))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)  # binary output

    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


# Binary Hyperparameter Tune

In [ ]:
def run_binary_hyperparameter_tuning(binary_file_paths, binary_labels):
    """Run hyperparameter tuning for binary classification model."""
    TUNER_DIR = "tuner_results_binary"

    # Split binary dataset
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        binary_file_paths,
        binary_labels,
        test_size=0.2,
        stratify=binary_labels,
        random_state=SEED
    )

    # Create datasets using the same hybrid pipeline
    train_ds = create_hybrid_dataset(train_paths, train_labels, BATCH_SIZE, augment=True)
    val_ds = create_hybrid_dataset(val_paths, val_labels, BATCH_SIZE, augment=False)

    # Tuner setup for binary model
    tuner = kt.Hyperband(
        build_binary_model,
        objective='val_accuracy',
        max_epochs=20,
        factor=3,
        hyperband_iterations=3,
        directory=TUNER_DIR,
        project_name='binary_cnn_finetune',
        overwrite=False,
        seed=SEED
    )

    # Callbacks
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    )

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )

    # Run the search
    tuner.search(
        train_ds,
        validation_data=val_ds,
        epochs=20,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )

    # Retrieve best hyperparameters
    best_hp = tuner.get_best_hyperparameters(1)[0]
    print("\nBest Hyperparameters for Binary Model:")
    for param in best_hp.values:
        print(f"{param}: {best_hp.get(param)}")

    return best_hp


# Binary Model Train

In [ ]:
def train_binary_model(best_hp, binary_file_paths, binary_labels):
    """Train binary classification model using standard train/val split."""
    BINARY_MODEL_PATH = "binary_model/binary_classifier_best.keras"
    os.makedirs(os.path.dirname(BINARY_MODEL_PATH), exist_ok=True)

    # Split binary data
    train_paths, val_paths, train_labels, val_labels = train_test_split(
        binary_file_paths,
        binary_labels,
        test_size=0.2,
        stratify=binary_labels,
        random_state=SEED
    )

    # Compute class weights
    class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

    # Create datasets
    train_ds = create_hybrid_dataset(train_paths, train_labels, BATCH_SIZE, augment=True)
    val_ds = create_hybrid_dataset(val_paths, val_labels, BATCH_SIZE, augment=False)

    # Build model
    model = build_binary_model(best_hp)

    # Callbacks
    callbacks = [
        EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_accuracy', patience=5, factor=0.5, min_lr=1e-5, verbose=1),
        ModelCheckpoint(BINARY_MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)
    ]

    # Train
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )

    print(f"\nTraining complete. Best model saved to: {BINARY_MODEL_PATH}")

    return model


# Run Binary Model tuning-training

In [11]:
binary_file_paths, binary_labels = get_binary_class_dataset(file_paths, labels)
binary_best_hp = run_binary_hyperparameter_tuning(binary_file_paths, binary_labels)

Trial 90 Complete [00h 00m 50s]
val_accuracy: 0.4907975494861603

Best val_accuracy So Far: 0.6319018602371216
Total elapsed time: 01h 06m 06s

Best Hyperparameters for Binary Model:
learning_rate: 0.00022517483664065638
dropout: 0.3
l2: 0.00040570101419583397
se_ratio: 8
se_dropout: 0.05
tuner/epochs: 20
tuner/initial_epoch: 7
tuner/bracket: 2
tuner/round: 2
tuner/trial_id: 0044


In [12]:
binary_model= train_binary_model(binary_best_hp, binary_file_paths, binary_labels)

Epoch 1/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step - accuracy: 0.7010 - loss: 0.7143
Epoch 1: val_accuracy improved from -inf to 0.50920, saving model to /kaggle/working/binary_model/binary_classifier_best.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 30s 325ms/step - accuracy: 0.7018 - loss: 0.7127 - val_accuracy: 0.5092 - val_loss: 0.7605 - learning_rate: 2.2517e-04
Epoch 2/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.7618 - loss: 0.5700
Epoch 2: val_accuracy did not improve from 0.50920
41/41 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step - accuracy: 0.7626 - loss: 0.5690 - val_accuracy: 0.5092 - val_loss: 0.7576 - learning_rate: 2.2517e-04
Epoch 3/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8192 - loss: 0.4340
Epoch 3: val_accuracy improved from 0.50920 to 0.64417, saving model to /kaggle/working/binary_model/binary_classifier_best.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 4s 59ms/step - accuracy: 0.8197 - loss: 0.4337 - val_accuracy: 0.6442 - val_loss: 0.7552 - learning_rate: 2.2517e-04
Ep

#  Main Model

In [13]:

def build_model(hp):
    lr = hp.Float("learning_rate", 1e-4, 1e-3, sampling="log")
    dropout = hp.Float("dropout", 0.3, 0.6, step=0.1)
    l2_val = hp.Float("l2", 1e-5, 1e-3, sampling="log")
    
    # tunable hyperparams for attention
    se_ratio = hp.Choice("se_ratio", [8, 16])
    se_dropout = hp.Float("se_dropout", 0.05, 0.2, step=0.05)

    inputs = tf.keras.layers.Input(shape=(N_MELS, FIXED_TIME_STEPS, 1))

    # Initial block
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 1
    residual = x
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = squeeze_excite_block(x, ratio=se_ratio, dropout_rate=se_dropout)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 2
    residual = layers.Conv2D(64, (1, 1), padding='same')(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = squeeze_excite_block(x, ratio=se_ratio, dropout_rate=se_dropout)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 3
    residual = layers.Conv2D(128, (1, 1), padding='same')(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = squeeze_excite_block(x, ratio=se_ratio, dropout_rate=se_dropout)
    x = layers.MaxPooling2D((2, 2))(x)

    # Final Dense Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(l2_val))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


In [ ]:
# Dummy hyperparams to test
#from keras_tuner.engine.hyperparameters import HyperParameters

#hp = HyperParameters()
#hp.values = {
#    "learning_rate": 1e-4,
#    "dropout": 0.4,
#    "l2": 1e-4,
#    "se_ratio": 8,
#    "se_dropout": 0.1
}

#model = build_model(hp)
#model.summary()

# Main Model Tuner

In [ ]:
# TODO: Later migth add batch size tuning (like [16, 32, 64]) since attention mechanisms can be sensitive to batch statistics

def run_hyperparameter_tuning():
    """Run hyperparameter tuning with hybrid dataset approach"""
    TUNER_DIR = "tuner_results"
    
    # Split data
    tune_train_paths, tune_val_paths, tune_train_labels, tune_val_labels = train_test_split(
        file_paths,
        labels,
        test_size=0.2,
        stratify=labels,
        random_state=SEED
    )
    
    # Create datasets
    tune_train_ds = create_hybrid_dataset(tune_train_paths, tune_train_labels, BATCH_SIZE, augment=True)
    tune_val_ds = create_hybrid_dataset(tune_val_paths, tune_val_labels, BATCH_SIZE, augment=False)
    
    # Tuner setup
    tuner = kt.Hyperband(
        build_model,
        objective='val_accuracy',
        max_epochs=20,
        factor=3,
        hyperband_iterations=2,  # Can be incresed to 3
        directory=TUNER_DIR,
        project_name='mel_cnn_finetune',
        overwrite=False,
        seed=SEED
    )

    # Callbacks
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    )
    
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )

    # Run search
    tuner.search(
        tune_train_ds,
        validation_data=tune_val_ds,
        epochs=20,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    
    # Best hyperparams
    best_hp = tuner.get_best_hyperparameters(1)[0]
    print("\nBest Hyperparameters Found:")
    for param in best_hp.values:
        print(f"{param}: {best_hp.get(param)}")
        
    return best_hp


# Train

In [ ]:
# Updated K-Fold training with hybrid dataset
def train_kfold(best_hp):
    """Train with k-fold cross-validation using hybrid datasets"""
    MODEL_DIR = os.path.join(os.getcwd(), "model")
    os.makedirs(MODEL_DIR, exist_ok=True)
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(file_paths, labels)):
        print(f"\n Starting Fold {fold + 1}/{N_FOLDS}")
        
        # Split data for this fold
        train_files = file_paths[train_idx]
        val_files = file_paths[val_idx]
        train_labels_fold = labels[train_idx]
        val_labels_fold = labels[val_idx]
        
        # Compute class weights
        class_weights = compute_class_weight('balanced', classes=np.unique(train_labels_fold), y=train_labels_fold)
        class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
        
        # Create hybrid datasets
        train_ds = create_hybrid_dataset(train_files, train_labels_fold, BATCH_SIZE, augment=True)
        val_ds = create_hybrid_dataset(val_files, val_labels_fold, BATCH_SIZE, augment=False)
        
        # Build model with best hyperparams
        model = build_model(best_hp)
        
        # Define callbacks
        model_path = f"model/fold_{fold + 1}_best_model.keras"
        callbacks = [
            EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_accuracy', patience=5, factor=0.5, min_lr=1e-5, verbose=1),
            ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True,verbose=1)
        ]
        
        # Train
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            class_weight=class_weight_dict,
            callbacks=callbacks,
            verbose=1
        )
        
        # Record best fold metrics
        best_val_acc = max(history.history['val_accuracy'])
        best_val_loss = min(history.history['val_loss'])
        
        print(f" Fold {fold + 1}: Best Val Accuracy = {best_val_acc:.4f}, Loss = {best_val_loss:.4f}")
        fold_results.append({
            "fold": fold + 1,
            "val_accuracy": best_val_acc,
            "val_loss": best_val_loss,
            "model_path": model_path
        })
    
    return fold_results
# Example usage:
# best_hp = run_hyperparameter_tuning()
# fold_results = train_kfold(best_hp)

# Tuning Begins

In [ ]:
# Hyperparameter tuning
best_hp = run_hyperparameter_tuning()

Trial 14 Complete [00h 01m 59s]
val_accuracy: 0.6749292016029358

Best val_accuracy So Far: 0.6749292016029358
Total elapsed time: 00h 20m 43s

Search: Running Trial #15

Value             |Best Value So Far |Hyperparameter
0.00013955        |0.00095138        |learning_rate
0.4               |0.4               |dropout
0.00029044        |0.00067187        |l2
8                 |8                 |se_ratio
0.2               |0.2               |se_dropout
7                 |7                 |tuner/epochs
3                 |3                 |tuner/initial_epoch
2                 |2                 |tuner/bracket
1                 |1                 |tuner/round
0008              |0002              |tuner/trial_id

Epoch 4/7
177/177 ━━━━━━━━━━━━━━━━━━━━ 49s 176ms/step - accuracy: 0.6739 - loss: 1.0096 - val_accuracy: 0.6190 - val_loss: 1.1860 - learning_rate: 1.3955e-04
Epoch 5/7
177/177 ━━━━━━━━━━━━━━━━━━━━ 22s 109ms/step - accuracy: 0.7402 - loss: 0.8330 - val_accuracy: 0.7139 - val_l

# Train Begins

In [ ]:
# Training
fold_results = train_kfold(best_hp)

# Test


---

### **Testing Pipeline Overview**

1. **Load Test Data**  
   Test audio files and labels are loaded, and mel spectrograms are cached (if not already).

2. **Model Ensemble Prediction**  
   - Each of the 5 fold models predicts the test set.  
   - Their predictions are combined via **majority voting** to get the **ensemble prediction**.

3. **Binary Classifier Patch** *(for class 5 vs 7 confusion)*  
   - From the ensemble, samples predicted as **class 7** are passed to a **binary classifier**.
   - If the binary model predicts **class 5**, the ensemble prediction is overridden to 5.

4. **Final Evaluation**  
   - Accuracy and classification report are computed **before and after** patching.
   - The pipeline compares **best single model**, **ensemble**, and **patched results**.

---



## TODO:

## Optimization Ideas

1. **Confidence-Based Patching**:
   - Consider only patching predictions where the ensemble is less confident (e.g., probability < 0.8)
   - This could prevent "correcting" high-confidence correct predictions

2. **Binary Model Threshold Tuning**:
   - You could find the optimal threshold for your binary classifier using ROC analysis
   - A simple validation set could help tune this threshold parameter

3. **Error Analysis Visualization**:
   - Consider adding confusion matrix visualization
   - Potentially include sample-level inspection of examples where patching changes the prediction

4. **Potential Weight Adjustments**:
   - Weight fold models by their validation performance when averaging probabilities
   - This could give more influence to better-performing models

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import librosa

# ========== Paths ==========
TEST_CACHE_DIR = "dataset/test_mel_cache"
BINARY_MODEL_PATH = "binary_model/best_model.h5"
MODEL_DIR = "model"

# ========== Create Cache Directory ==========
os.makedirs(TEST_CACHE_DIR, exist_ok=True)

# ========== Load Test Data ==========
test_df = pd.read_csv(TEST_CSV_PATH)
test_df["filepath"] = test_df["file_name"].apply(lambda x: os.path.join(TEST_AUDIO_DIR, x))
test_paths = test_df["filepath"].values
test_labels = test_df["classID"].values

# ========== Audio Loader ==========
def load_and_cache_audio_test(file_path):
    cache_file = os.path.join(TEST_CACHE_DIR, os.path.basename(file_path).replace('.wav', '.npy'))
    if os.path.exists(cache_file):
        try:
            return np.load(cache_file)
        except:
            pass
    try:
        y, sr = librosa.load(file_path, sr=SR)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, fmax=FMAX)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        mel_db = np.expand_dims(mel_db, axis=-1)
        np.save(cache_file, mel_db)
        return mel_db
    except:
        return np.zeros((N_MELS, FIXED_TIME_STEPS, 1))

# ========== TF Dataset Builder ==========
def create_test_dataset(file_paths, labels, batch_size=32):
    def process_path(file_path, label):
        def _load(path):
            path = path.numpy().decode("utf-8")
            mel = load_and_cache_audio_test(path)
            if mel.shape[1] != FIXED_TIME_STEPS:
                mel = librosa.util.fix_length(np.squeeze(mel, axis=-1), FIXED_TIME_STEPS, axis=1)
                mel = np.expand_dims(mel, axis=-1)
            elif mel.ndim == 2:
                mel = np.expand_dims(mel, axis=-1)
            return mel.astype(np.float32)
        mel = tf.py_function(_load, inp=[file_path], Tout=tf.float32)
        mel = tf.ensure_shape(mel, [N_MELS, FIXED_TIME_STEPS, 1])
        return mel, tf.cast(label, tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# ========== Load Binary Classifier ==========
binary_model = tf.keras.models.load_model(BINARY_MODEL_PATH)

def patch_predictions(preds, file_paths, threshold=0.5):
    idx_7 = np.where(preds == 7)[0]
    if len(idx_7) == 0:
        return preds
    patch_ds = create_test_dataset(file_paths[idx_7], np.zeros(len(idx_7)))
    probs = binary_model.predict(patch_ds, verbose=0).squeeze()
    binary_preds = (probs >= threshold).astype(int)
    patched = preds.copy()
    patched[idx_7[binary_preds == 1]] = 5
    return patched

# ========== Load Fold Models and Predict ==========
model_paths = sorted([
    os.path.join(MODEL_DIR, f) for f in os.listdir(MODEL_DIR)
    if f.endswith(".keras") and f.startswith("fold_")
])

print("\n Evaluating models...\n")
soft_preds = []
hard_preds = []
patched_hard_preds = []
patched_soft_preds = []
accuracies = []

test_ds = create_test_dataset(test_paths, test_labels)

for i, model_path in enumerate(model_paths):
    print(f"→ Loading model: {model_path}")
    model = tf.keras.models.load_model(model_path)
    probs = model.predict(test_ds, verbose=0)
    
    # Hard prediction
    hard = np.argmax(probs, axis=1)
    acc = accuracy_score(test_labels, hard)
    accuracies.append(acc)
    hard_preds.append(hard)

    # Patch hard preds
    patched = patch_predictions(hard, test_paths)
    patched_hard_preds.append(patched)

    # Store soft preds
    soft_preds.append(probs)

    print(f"   Fold Accuracy: {acc:.4f}")

# ========== Ensemble Voting ==========
soft_preds = np.stack(soft_preds)  # (n_models, n_samples, n_classes)
avg_probs = np.mean(soft_preds, axis=0)  # (n_samples, n_classes)
ensemble_preds = np.argmax(avg_probs, axis=1)
ensemble_acc = accuracy_score(test_labels, ensemble_preds)

print("\n Ensemble Classification Report:")
print(classification_report(test_labels, ensemble_preds, digits=4))
print(f"Ensemble Accuracy: {ensemble_acc:.4f}")

# ========== Patched Ensemble ==========
patched_ensemble_preds = patch_predictions(ensemble_preds, test_paths)
patched_acc = accuracy_score(test_labels, patched_ensemble_preds)

print("\n Patched Ensemble Classification Report:")
print(classification_report(test_labels, patched_ensemble_preds, digits=4))
print(f"Patched Ensemble Accuracy: {patched_acc:.4f}")

# ========== Individual Fold Comparisons ==========
print("\n === Individual Fold Summaries ===\n")
for i, (raw, patched, acc) in enumerate(zip(hard_preds, patched_hard_preds, accuracies)):
    patched_acc = accuracy_score(test_labels, patched)
    print(f"Fold {i+1}:")
    print(f"  Raw    Accuracy: {acc:.4f}")
    print(f"  Patched Accuracy: {patched_acc:.4f}")
    print("")

# ========== Summary: Best Accuracy Across All Predictions ==========
all_results = {
    "Ensemble": ensemble_preds,
    "Patched Ensemble": patched_ensemble_preds,
}

# Add folds and patched folds
for i, preds in enumerate(hard_preds):
    all_results[f"Fold {i+1}"] = preds
for i, preds in enumerate(patched_hard_preds):
    all_results[f"Fold {i+1} (Patched)"] = preds

# Find best
best_name = None
best_acc = -1
best_preds = None

for name, preds in all_results.items():
    acc = accuracy_score(test_labels, preds)
    if acc > best_acc:
        best_acc = acc
        best_name = name
        best_preds = preds

print(f"\n Best Result: {best_name}")
print(f"Accuracy: {best_acc:.4f}")
print("Classification Report:")
print(classification_report(test_labels, best_preds, digits=4))


### Test With Google Drive  **update requaired

In [ ]:
"""
Tarık Buğra Ay - 042101100
"""

import os
import io
import zipfile
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import gdown
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

# ========== CONFIG ==========
MODEL_ZIP_URL = "https://drive.google.com/uc?export=download&id=1f3Mh5HiuIcuACEgqY9qcbYg9WyopmJWY"
TEST_CSV_PATH = "/kaggle/input/yldz-teknik-proje-1-train-dataset/Test_Public.csv"
TEST_AUDIO_DIR = "/kaggle/input/yldz-teknik-proje-1-train-dataset/Test_Public/"
SR = 22050
N_MELS = 128
FMAX = 8000
FIXED_TIME_STEPS = 150
BATCH_SIZE = 32
EXTRACT_DIR = "/kaggle/working/extracted_models"
# ============================

# Step 1: Download and extract ZIP
print(" Downloading model ZIP from Google Drive...")
zip_path = "models.zip"
gdown.download(MODEL_ZIP_URL, zip_path, quiet=False)

print(" Extracting models from ZIP...")
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

# Step 2: Load all .keras models
models = []
model_filenames = sorted([
    f for f in os.listdir(EXTRACT_DIR)
    if f.endswith(".keras") and f.startswith("fold_")
])

print(f" Found {len(model_filenames)} models: {model_filenames}\n")

for name in model_filenames:
    model_path = os.path.join(EXTRACT_DIR, name)
    print(f"→ Loading model: {model_path}")
    model = tf.keras.models.load_model(model_path)
    models.append(model)

# Step 3: Load and process test dataset
print("\n Loading and preprocessing test set...")
df = pd.read_csv(TEST_CSV_PATH)
file_paths = [os.path.join(TEST_AUDIO_DIR, fname) for fname in df["file_name"]]
true_labels = df["classID"].values

def preprocess_audio(file_path):
    try:
        y, sr = librosa.load(file_path, sr=SR)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, fmax=FMAX)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        mel_db = np.expand_dims(mel_db, axis=-1)

        if mel_db.shape[1] != FIXED_TIME_STEPS:
            mel_db = librosa.util.fix_length(mel_db.squeeze(), size=FIXED_TIME_STEPS, axis=1)
            mel_db = np.expand_dims(mel_db, axis=-1)

        return mel_db.astype(np.float32)
    except Exception as e:
        print(f"[ERROR] Could not process {file_path}: {e}")
        return np.zeros((N_MELS, FIXED_TIME_STEPS, 1), dtype=np.float32)

X_test = np.stack([preprocess_audio(p) for p in tqdm(file_paths)])
y_test = np.array(true_labels)

# Step 4: Evaluate each model
print("\n Evaluating models...\n")
model_preds = []
accuracies = []

for i, model in enumerate(models):
    print(f"→ Evaluating Model {i+1}/{len(models)}")
    probs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    preds = np.argmax(probs, axis=1)

    acc = accuracy_score(y_test, preds)
    model_preds.append(preds)
    accuracies.append(acc)

    print(f"   Accuracy: {acc:.4f}")

# Step 5: Ensemble prediction
print("\nPerforming ensemble prediction (majority vote)...")
model_preds = np.stack(model_preds)  # Shape: (n_models, n_samples)
ensemble_preds = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=model_preds)
ensemble_acc = accuracy_score(y_test, ensemble_preds)

# Step 6: Report
print(f"\nEnsemble Accuracy: {ensemble_acc:.4f}")
print("\nClassification Report (Ensemble):")
print(classification_report(y_test, ensemble_preds, digits=4))

# Step 7: Best model info
best_model_idx = np.argmax(accuracies)
print("\n Best Single Model:")
print(f"   Model File : {model_filenames[best_model_idx]}")
print(f"   Accuracy   : {accuracies[best_model_idx]:.4f}")

if ensemble_acc > accuracies[best_model_idx]:
    print("Ensemble outperformed the best individual model.")
else:
    print("Best individual model outperformed the ensemble.")

# Clean up ZIP file
if os.path.exists(zip_path):
    os.remove(zip_path)


# Investigating

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print("\nClass 5 sample count in each fold:")

for fold, (train_idx, val_idx) in enumerate(skf.split(file_paths, labels)):
    train_labels_fold = labels[train_idx]
    count_class_5 = np.sum(train_labels_fold == 5)
    print(f"Fold {fold+1}: {count_class_5} samples in training")


In [ ]:
# Show predictions for class 5
print("\n Class 5 Test Sample Predictions:")
class_5_indices = np.where(y_test == 5)[0]

for i in class_5_indices:
    fname = os.path.basename(file_paths[i])
    pred = ensemble_preds[i]
    print(f"{fname:20} | Predicted: {pred}")


In [ ]:
import IPython.display as ipd
import random

def play_random_class_audios(class_id, n=4):
    """Play n random audio files from a given class."""
    class_df = df[df["classID"] == class_id]
    if len(class_df) == 0:
        print(f"No samples found for class {class_id}")
        return

    selected = class_df.sample(n=min(n, len(class_df)), random_state=np.random.randint(0, 10000))
    
    for i, row in enumerate(selected.itertuples()):
        print(f"\n Playing Class {class_id} - {os.path.basename(row.filepath)}")
        audio, _ = librosa.load(row.filepath, sr=SR)
        ipd.display(ipd.Audio(audio, rate=SR))


In [ ]:
play_random_class_audios(5)


In [ ]:
play_random_class_audios(7)
